# 06 – SHAP Analysis of Learned Parameters

Uses DeepSHAP to explain which landscape attributes most influence the
dPL-SPARROW parameter generators.

Three sub-networks are explained separately:

| Sub-network | Inputs | Output explained |
|---|---|---|
| `param_model` | 9 catchment attributes | α (N export rate) |
| `param_model_strm` | slope, meanq | θ_S (stream attenuation) |
| `param_model_res` | meanTemp | θ_R (reservoir attenuation) |

Requires: `pip install shap`


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

from sparrow import SPARROW, ParamGenerator, hydseq, setup_seed

setup_seed(42)
device = torch.device('cpu')   # DeepSHAP runs on CPU for stability

# ── Paths (edit these to point to your data) ─────────────────────────────────
# sparrow_input_path : same file as in notebook 01 (see column list there)
# trendn_path        : same TREND-N file as in notebook 01 (see column list there)
sparrow_input_path = 'data/sparrow_input.csv'               # <-- CHANGE THIS
trendn_path        = 'data/trendn_surplus.csv'              # <-- CHANGE THIS
checkpoint_path    = 'outputs/checkpoint_epoch_150.pth'   # trained checkpoint
SEED = 42


## Load data and fit scalers

In [ ]:
from sklearn.preprocessing import MinMaxScaler

input_data_huc12 = pd.read_csv(sparrow_input_path)
input_data_huc12['deliver_0'] = 0

surplus = pd.read_csv(trendn_path)
surplus['waterid'] = surplus['waterid'].astype(int)
input_data_huc12 = input_data_huc12.merge(surplus, on='waterid', how='left').fillna(0)

input_data_huc12['area_ha'] = input_data_huc12['demiarea'] * 100
input_data_huc12['N_surplus'] = (
    input_data_huc12[['Atmospheric_Oxidized','Atmospheric_Reduced',
                       'Fertilizer_Agriculture','Fertilizer_NonAgriculture',
                       'Fix_Cropland','Fix_Pasture','Human',
                       'Lvst_BeefCow','Lvst_Broilers','Lvst_DairyCow','Lvst_Equine',
                       'Lvst_Hogs','Lvst_LayersPullets','Lvst_OtherCattle',
                       'Lvst_SheepGoat','Lvst_Turkeys']].sum(axis=1) -
    input_data_huc12[['CropUptake_Cropland','CropUptake_Pasture']].sum(axis=1))
input_data_huc12['total_N_surplus'] = input_data_huc12['N_surplus'] * input_data_huc12['area_ha']

input_columns      = ['PPT', 'tiles_perc', 'soil_CLAYAVE', 'meanTemp',
                       'CRP_percent', 'no_till', 'cover_crop_percent',
                       'forest_percent', 'wetlands_percent']
input_columns_strm = ['slope', 'meanq']
input_columns_res  = ['meanTemp']

delivery_columns  = ['deliver_0']
stream_columns    = ['strmloss']
reservoir_columns = ['iresload']

scaler      = MinMaxScaler().fit(input_data_huc12[input_columns])
scaler_strm = MinMaxScaler().fit(input_data_huc12[input_columns_strm])
scaler_res  = MinMaxScaler().fit(input_data_huc12[input_columns_res])


## Load trained checkpoint

In [ ]:
param_model = ParamGenerator(
    input_size=len(input_columns), hidden_size=32,
    num_source=1, num_delivery=len(delivery_columns),
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)
)

param_model_strm = ParamGenerator(
    input_size=len(input_columns_strm), hidden_size=8,
    num_source=1, num_delivery=len(delivery_columns),
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)
)

param_model_res = ParamGenerator(
    input_size=len(input_columns_res), hidden_size=8,
    num_source=1, num_delivery=len(delivery_columns),
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)
)

ckpt = torch.load(checkpoint_path, map_location='cpu')
param_model.load_state_dict(ckpt['model_state_dict'])
param_model_strm.load_state_dict(ckpt['param_model_strm_state_dict'])
param_model_res.load_state_dict(ckpt['param_model_res_state_dict'])
param_model.eval(); param_model_strm.eval(); param_model_res.eval()
print('Checkpoint loaded.')


## Prepare scaled features (all reach-years)

In [ ]:
X_all      = torch.tensor(scaler.transform(input_data_huc12[input_columns]),
                           dtype=torch.float32)
X_all_strm = torch.tensor(scaler_strm.transform(input_data_huc12[input_columns_strm]),
                           dtype=torch.float32)
X_all_res  = torch.tensor(scaler_res.transform(input_data_huc12[input_columns_res]),
                           dtype=torch.float32)

# Thin down to unique HUC12s (average over years) for SHAP background
input_data_huc12['HUC12'] = input_data_huc12['waterid'].astype(str).str[:-2]
huc12_mean = input_data_huc12.groupby('HUC12')[
    input_columns + input_columns_strm].mean().reset_index()

X_bg      = torch.tensor(scaler.transform(huc12_mean[input_columns]),     dtype=torch.float32)
X_bg_strm = torch.tensor(scaler_strm.transform(huc12_mean[input_columns_strm]), dtype=torch.float32)
X_bg_res  = torch.tensor(scaler_res.transform(huc12_mean[input_columns_res]),   dtype=torch.float32)

rng = np.random.default_rng(SEED)
n_explain = min(20000, len(huc12_mean))
idx       = rng.choice(len(huc12_mean), n_explain, replace=False)

X_explain      = X_bg[idx]
X_explain_strm = X_bg_strm[idx]
X_explain_res  = X_bg_res[idx]
print(f'Background: {len(X_bg)} HUC12s | Explaining: {n_explain} samples')


## DeepSHAP

Each SHAP analysis targets the primary output of its sub-network:
- `param_model` → output 0 (α, N export rate)
- `param_model_strm` → output −2 (θ_S, stream attenuation)
- `param_model_res`  → output −1 (θ_R, reservoir attenuation)


In [ ]:
import shap

# ── Helper: single-output wrapper ─────────────────────────────────────────────
class SingleOutput(nn.Module):
    def __init__(self, base, out_idx):
        super().__init__()
        self.base = base
        self.out_idx = out_idx
    def forward(self, x):
        return self.base(x)[:, self.out_idx:self.out_idx+1]

# ── catchment model: N export rate (alpha) ────────────────────────────────────
model_alpha   = SingleOutput(param_model, 0)
explainer_cat = shap.DeepExplainer(model_alpha, X_bg)
shap_alpha    = explainer_cat.shap_values(X_explain)
if isinstance(shap_alpha, list): shap_alpha = shap_alpha[0]

# ── stream model: stream attenuation (theta_S) ───────────────────────────────
model_thS    = SingleOutput(param_model_strm, -2)
explainer_strm = shap.DeepExplainer(model_thS, X_bg_strm)
shap_thS     = explainer_strm.shap_values(X_explain_strm)
if isinstance(shap_thS, list): shap_thS = shap_thS[0]

# ── reservoir model: reservoir attenuation (theta_R) ─────────────────────────
model_thR    = SingleOutput(param_model_res, -1)
explainer_res = shap.DeepExplainer(model_thR, X_bg_res)
shap_thR     = explainer_res.shap_values(X_explain_res)
if isinstance(shap_thR, list): shap_thR = shap_thR[0]

print('SHAP values computed.')
print(f'  alpha  shape: {np.array(shap_alpha).shape}')
print(f'  theta_S shape: {np.array(shap_thS).shape}')
print(f'  theta_R shape: {np.array(shap_thR).shape}')


## Feature importance summary

In [ ]:
def importance_df(shap_vals, feature_names, param_name):
    imp = np.abs(shap_vals).mean(axis=0)
    return pd.DataFrame({'feature': feature_names,
                          'mean_abs_shap': imp,
                          'parameter': param_name}).sort_values('mean_abs_shap', ascending=False)

df_imp_alpha = importance_df(shap_alpha, input_columns,      'alpha (N export rate)')
df_imp_thS   = importance_df(shap_thS,   input_columns_strm, 'theta_S (stream loss)')
df_imp_thR   = importance_df(shap_thR,   input_columns_res,  'theta_R (reservoir loss)')

df_imp_all = pd.concat([df_imp_alpha, df_imp_thS, df_imp_thR], ignore_index=True)
print(df_imp_all.to_string(index=False))

output_dir = 'outputs/'
os.makedirs(output_dir, exist_ok=True)
df_imp_all.to_csv(output_dir + 'shap_feature_importance.csv', index=False)

# Save raw SHAP arrays
np.save(output_dir + 'shap_alpha.npy',   np.array(shap_alpha))
np.save(output_dir + 'shap_theta_S.npy', np.array(shap_thS))
np.save(output_dir + 'shap_theta_R.npy', np.array(shap_thR))
print('Saved SHAP results to', output_dir)
